In [ ]:
import pandas as pd

# Load dataset
df = pd.read_csv("customer_support_tickets.csv")

# Check data
print(df.head())
print(df.info())

# Convert relevant columns to datetime objects
df['First Response Time'] = pd.to_datetime(df['First Response Time'])
df['Time to Resolution'] = pd.to_datetime(df['Time to Resolution'])

# Calculate 'Resolution Duration' in hours, which will be the target variable.
# This will be NaN where either of the original columns was NaN.
df['Resolution Duration'] = (df['Time to Resolution'] - df['First Response Time']).dt.total_seconds() / 3600

# Drop irrelevant columns and the original datetime columns after creating the duration target.
df = df.drop(columns=[
    "Ticket ID",
    "Customer Name",
    "Customer Email",
    "Ticket Description",
    "Date of Purchase",      # Assuming 'Date of Purchase' is not needed as a direct feature
    "Resolution",            # Assuming 'Resolution' text is not needed as a direct feature
    "Time to Resolution",    # Dropped after creating 'Resolution Duration'
    "First Response Time"    # Dropped after creating 'Resolution Duration'
])

# Print null sums to observe changes after dropping columns and before filling/dropping NaNs
print(df.isnull().sum())

# Fill missing customer satisfaction ratings
df["Customer Satisfaction Rating"].fillna(df["Customer Satisfaction Rating"].mean(), inplace=True)

# Drop rows where 'Resolution Duration' is NaN, as these tickets are either unresolved
# or their duration cannot be calculated, making them unsuitable for training this model.
df.dropna(subset=['Resolution Duration'], inplace=True)

# Separate features (X) and target (y) BEFORE one-hot encoding the features.
X = df.drop("Resolution Duration", axis=1)
y = df["Resolution Duration"]

# Apply one-hot encoding to features (X) only.
X = pd.get_dummies(X, drop_first=True)


from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)


from sklearn.metrics import mean_absolute_error, mean_squared_error

print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))



from sklearn.ensemble import RandomForestRegressor

# Create model
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=5,
    random_state=42
)

# Train
rf_model.fit(X_train, y_train)

# Predict
y_pred_rf = rf_model.predict(X_test)

# Evaluate
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("Random Forest MAE:", mean_absolute_error(y_test, y_pred_rf))
print("Random Forest MSE:", mean_squared_error(y_test, y_pred_rf))

print("Linear Regression MAE:", mean_absolute_error(y_test, y_pred))
print("Random Forest MAE:", mean_absolute_error(y_test, y_pred_rf))


from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

print("MAE:", mean_absolute_error(y_test, y_pred_rf))
print("MSE:", mean_squared_error(y_test, y_pred_rf))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_rf)))
print("R2 Score:", r2_score(y_test, y_pred_rf))


   Ticket ID        Customer Name              Customer Email  Customer Age  \
0          1        Marisa Obrien  carrollallison@example.com            32   
1          2         Jessica Rios    clarkeashley@example.com            42   
2          3  Christopher Robbins   gonzalestracy@example.com            48   
3          4     Christina Dillon    bradleyolson@example.org            27   
4          5    Alexander Carroll     bradleymark@example.com            67   

  Customer Gender Product Purchased Date of Purchase      Ticket Type  \
0           Other        GoPro Hero       2021-03-22  Technical issue   
1          Female       LG Smart TV       2021-05-22  Technical issue   
2           Other          Dell XPS       2020-07-14  Technical issue   
3          Female  Microsoft Office       2020-11-13  Billing inquiry   
4          Female  Autodesk AutoCAD       2020-02-04  Billing inquiry   

             Ticket Subject  \
0             Product setup   
1  Peripheral compatibil

/tmp/ipykernel_2438/2560482152.py:34: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Customer Satisfaction Rating"].fillna(df["Customer Satisfaction Rating"].mean(), inplace=True)


Random Forest MAE: 7.730667854932945
Random Forest MSE: 91.7638789893963
Linear Regression MAE: 7.884243874328805
Random Forest MAE: 7.730667854932945
MAE: 7.730667854932945
MSE: 91.7638789893963
RMSE: 9.579346480287489
R2 Score: -0.00903381648925805
